## ***Initials***

In [26]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import json
from tqdm import tqdm
import numpy as np
import ollama

## ***Help Functions***

In [27]:
# function that: creates the LLM RAG prompt
def format_llm_prompt(top3_df, nace_code, nace_description, user_query=None):
    user_context = f'The user wants to open a business of type: "{user_query}"\n\n' if user_query else ""
    
    nace_info = f"**NACE Code: {nace_code} — {nace_description}**\n\n"
    
    neighborhood_block = "Based on data analysis, here are the top 3 recommended neighborhoods in Volos:\n\n"
    for i, row in top3_df.iterrows():
        neighborhood_block += f"{i+1}. **{row['Neighborhood']}**\n   {row['Description']}\n\n"

    instruction = (
        "Please present the 3 neighborhoods in a numbered list format like this:\n"
        "1. Neighborhood Name: Summary of strengths...\n"
        "2. Neighborhood Name: Summary of strengths...\n"
        "3. Neighborhood Name: Summary of strengths...\n\n"
        "Each point should be 2–4 sentences long, clearly explaining why the neighborhood is a good fit for the business. "
        "Avoid ranking or comparisons — describe each one positively and independently."
    )

    return user_context + nace_info + neighborhood_block + instruction

In [28]:
# === Embed function ===
def get_embedding(text, tokenizer, model, device):
    prompt = "Represent this sentence for retrieval: " + text
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(device)
    with torch.no_grad():
        output = model(**inputs)
        embedding = output.last_hidden_state[:, 0]
        embedding = F.normalize(embedding, p=2, dim=1)
    return embedding.cpu().numpy()[0]

## ***Load the Data***

### ***NACE Codes***

In [29]:
# === Load the data ===
df_naces = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\2. General Data - Directories\\3. NACE Data\\unique_naces.csv")
nace_dict = dict(zip(df_naces['NACE Code'].astype(str), df_naces['Class Description']))
print(len(df_naces))
df_naces.head()

307


,NACE Code,Class Description
0,1.13,"Growing of vegetables and melons, roots and tu..."
1,1.30,Plant propagation
2,1.49,Raising of other animals
3,1.61,Support activities for crop production
4,2.40,Support services to forestry


### ***Top-10 Neighborhoods per NACE***

In [30]:
top10_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\6. Ground Truths\\4. Ensembling of Approaches\\Extracted CSV Files\\top10_final.csv")
print(top10_df.shape)
print(top10_df.columns)
top10_df.head()

(306, 11)
Index(['NACE Code', 'Top1', 'Top2', 'Top3', 'Top4', 'Top5', 'Top6', 'Top7',
       'Top8', 'Top9', 'Top10'],
      dtype='object')


,NACE Code,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,Top9,Top10
0,1.13,Metamorfosi,Kato Lehonia,Palaia,Epta Platania - Oksigono,Agioi Anargiroi,Agia Paraskeui,Agios Konstantinos,Agios Vasilios,Neapoli,Analipsi
1,1.30,Agia Paraskeui,Epta Platania - Oksigono,Neapoli,Agios Vasilios,Palaia,Nea Ionia,Agios Konstantinos,Asteria Agrias,Agia Kyriaki,Aivaliotika
2,1.49,Nees Pagases,Dimini,Melissatika,Agios Georgios,Neapoli,Epta Platania - Oksigono,Aivaliotika,Agios Vasilios,Alli Meria,Agia Paraskeui
3,1.61,Nees Pagases,Epta Platania - Oksigono,Dimitriada,Aivaliotika,Metamorfosi,Agia Paraskeui,Agios Georgios,Analipsi,Dimini,Agios Konstantinos
4,2.40,Metamorfosi,Analipsi,Asteria Agrias,Agios Konstantinos,Epta Platania - Oksigono,Agia Paraskeui,Agia Kyriaki,Palaia,Nees Pagases,Aivaliotika


### ***Neighborhood Descriptions***

In [32]:
neighborhood_descriptions_df = pd.read_csv("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\8. LLM - RAG\\Extracted Files\\Neighborhood Descriptions\\neighborhood_descriptions_20.csv")
print(neighborhood_descriptions_df.shape)
print(neighborhood_descriptions_df.columns)
neighborhood_descriptions_df.head()

(48, 2)
Index(['Neighborhood', 'Description'], dtype='object')


,Neighborhood,Description
0,Dimini,Dimini – a spacious neighborhood (37.34 km²) w...
1,Xrisi Akti Panagias,Xrisi Akti Panagias – a low-density location (...
2,Sesklo,Sesklo – a low-density location (27.37 km²) th...
3,Agioi Anargiroi,Agioi Anargiroi – a cozy location (0.77 km²) t...
4,Aivaliotika,Aivaliotika – a neighborhood (4.85 km²) that o...


### ***Neighborhood Descriptions Embeddings***

In [33]:
# Load precomputed NACE embeddings
with open("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\8. LLM - RAG\\Extracted Files\\Neighborhood Description Embeddings\\neighborhoods_descriptions_embeddings_20.json", "r", encoding="utf-8") as f:
    precomputed_neighborhoods_embeddings = json.load(f)

### ***NACE Descriptions Embeddings***

In [34]:
# Load precomputed NACE embeddings
with open("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\8. LLM - RAG\\Extracted Files\\nace_descriptions_embeddings.json", "r", encoding="utf-8") as f:
    precomputed_nace_embeddings = json.load(f)

## ***Load the Embedder Model***

In [35]:
# Load the Embedder and its tokenizer
classifier_model_name = "BAAI/bge-large-en-v1.5"

In [36]:
classifier_tokenizer = AutoTokenizer.from_pretrained(classifier_model_name)
classifier_model = AutoModel.from_pretrained(classifier_model_name)

In [37]:
# Set device (CPU or CUDA if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier_model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 1024, padding_idx=0)
    (position_embeddings): Embedding(512, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-23): 24 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, 

# ***Inference: Single Query***

## ***Get the User Query and its Embedding***

In [12]:
query  = "Where in Volos should I start a bakery business?"

In [13]:
# === Embed user query ===
query_emb = get_embedding(query, classifier_tokenizer, classifier_model, device)

## ***Extract the NACE Code***

### ***Load the Embedded NACE Descriptions***

In [14]:
nace_codes = [item["nace_code"] for item in precomputed_nace_embeddings]
nace_embeddings = np.array([item["embedding"] for item in precomputed_nace_embeddings])

### ***Keep the most similar NACE Description***

In [15]:
# === Compute cosine similarities
sims = cosine_similarity([query_emb], nace_embeddings)[0]

In [16]:
# Get index of the highest similarity score
best_idx = np.argmax(sims)

# Extract best NACE code and description
nace_code = nace_codes[best_idx]
nace_description = df_naces[df_naces["NACE Code"] == nace_code]["Class Description"].values[0]

print(f"Predicted NACE Code: {nace_code} → {nace_description}")

Predicted NACE Code: 46.36 → Wholesale of sugar and chocolate and sugar confectionery


## ***Extract the Top-3 locations for this NACE Code***

### ***First Extract the top-10 locations of this NACE Code***

In [18]:
top10_of_nace = top10_df[top10_df['NACE Code'] == nace_code]
top10_names = top10_of_nace.iloc[0, 1:].tolist() if not top10_of_nace.empty else []

### ***Extract their Embeddings***

In [20]:
top10_neighborhoods_embeddings = [
    row for row in precomputed_neighborhoods_embeddings
    if row["neighborhood"] in top10_names
]

In [21]:
top10_neighborhood_descriptions_df = pd.DataFrame([{
        "Neighborhood": row["neighborhood"],
        "Embedding": np.array(row["embedding"])
    } for row in top10_neighborhoods_embeddings
])

In [22]:
top10_neighborhood_descriptions_df = pd.merge(top10_neighborhood_descriptions_df, neighborhood_descriptions_df, on="Neighborhood", how="left")

### ***Keep the most similar Neighborhood Description***

In [24]:
# Stack embeddings for similarity comparison
desc_embeddings = np.stack(top10_neighborhood_descriptions_df['Embedding'].values)

# Compute cosine similarity with the NACE description embedding
similarities = cosine_similarity([query_emb], desc_embeddings)[0]

# Add similarity scores
top10_neighborhood_descriptions_df['Similarity'] = similarities

# Sort and keep top 3
top3_df = top10_neighborhood_descriptions_df.sort_values(by='Similarity', ascending=False).head(3)
top3_df = top3_df[['Neighborhood', 'Description', 'Similarity']]

## ***Perform RAG***

In [26]:
# Construct the prompt for Llama3
prompt = format_llm_prompt(
    top3_df=top3_df,
    nace_code=nace_code,
    nace_description=nace_description,
    user_query=query
)

In [27]:
print(prompt)

The user wants to open a business of type: "Where in Volos should I start a bakery business?"

**NACE Code: 46.36 — Wholesale of sugar and chocolate and sugar confectionery**

Based on data analysis, here are the top 3 recommended neighborhoods in Volos:

5. **Hiliadou**
   Hiliadou – a balanced location (1.03 km²) — not too dense, not too open, you'll be right near the heart of Volos (1.17 km) — convenient for both customers and deliveries, very close to the port (2.00 km), which makes transport and logistics a lot easier, reasonably close to main roads (0.49 km), giving you decent accessibility without heavy traffic, a reasonable distance from transit (0.20 km) — close enough for most customers to reach you without much hassle, close to campus (1.33 km), which means steady foot traffic from a younger, educated crowd, lots of people live in the area — great for attracting walk-in customers, a neighborhood with many single adults (40%) — ideal for modern, fast-moving business models, f

In [28]:
# Call the Llama3 model with the constructed prompt
response = ollama.chat(model="llama3", messages=[{"role": "user", "content": prompt}])
print(response.message.content)

Here are the top 3 recommended neighborhoods in Volos:

1. **Hiliadou**: Hiliadou offers a balanced location that is neither too dense nor too open, making it an ideal spot for your bakery business. You'll be close to the heart of Volos, convenient for both customers and deliveries, and very near the port, which makes transport and logistics easier. Additionally, the neighborhood has a high percentage of single adults, making it a great fit for modern, fast-moving business models.

2. **Metamorfosi**: Metamorfosi is a cozy location that could work well for storefronts or quick-service ideas. You'll be right near the heart of Volos, convenient for both customers and deliveries, and conveniently located near the port, making it helpful for supply chains or frequent deliveries. The neighborhood also has a high percentage of single residents, which could favor trendy, flexible, or lifestyle-focused services.

3. **Karagats**: Karagats is a cozy location that might work well for storefronts

# ***Inference: On the Inference Dataset***

In [38]:
# === Load data files ===
with open("C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\7. LLM - Fine Tuning\\1. Fine-Tuning - Inference Datasets Creation\\3. Fine-Tuning - Inference Data Creation\\Extracted Files\\inference_dataset.jsonl", "r", encoding="utf-8") as f:
    inference_data = [json.loads(line) for line in f]

In [39]:
# === Output list
output_records = []

# === Inference loop
for example in tqdm(inference_data):
    query = example["prompt"]
    expected_completion = example["completion"]

    try:
        # ------------- Embed user query -------------
        query_emb = get_embedding(query, classifier_tokenizer, classifier_model, device)


        # ------------- NACE Code Extraction -------------
        # Load the precomputed NACE embeddings
        nace_codes = [item["nace_code"] for item in precomputed_nace_embeddings]
        nace_embeddings = np.array([item["embedding"] for item in precomputed_nace_embeddings])

        # Keep the most similar NACE code
        sims = cosine_similarity([query_emb], nace_embeddings)[0]
        best_idx = np.argmax(sims)
        nace_code = nace_codes[best_idx]
        nace_description = df_naces[df_naces["NACE Code"] == nace_code]["Class Description"].values[0]


        # ------------- Top-10 Candidates Extraction -------------
        # Extract the top 10 neighborhoods for this NACE
        top10_of_nace = top10_df[top10_df['NACE Code'] == nace_code]
        top10_names = top10_of_nace.iloc[0, 1:].tolist() if not top10_of_nace.empty else []



        # Extract the embeddings of the top 10 candidates
        top10_neighborhoods_embeddings = [
            row for row in precomputed_neighborhoods_embeddings
            if row["neighborhood"] in top10_names
        ]

        top10_neighborhood_descriptions_df = pd.DataFrame([{
                "Neighborhood": row["neighborhood"],
                "Embedding": np.array(row["embedding"])
            } for row in top10_neighborhoods_embeddings
        ])
        top10_neighborhood_descriptions_df = pd.merge(top10_neighborhood_descriptions_df, neighborhood_descriptions_df, on="Neighborhood", how="left")
        desc_embeddings = np.stack(top10_neighborhood_descriptions_df['Embedding'].values)


        # ------------- Top-3 Candidates Extraction -------------
        # Compute cosine similarity with the NACE description embedding
        similarities = cosine_similarity([query_emb], desc_embeddings)[0]
        top10_neighborhood_descriptions_df['Similarity'] = similarities

        # Sort and keep top 3
        top3_df = top10_neighborhood_descriptions_df.sort_values(by='Similarity', ascending=False).head(3)
        top3_df = top3_df[['Neighborhood', 'Description', 'Similarity']]


        # ------------- RAG Implementation -------------
        # Format prompt and call LLM
        prompt = format_llm_prompt(
            top3_df=top3_df,
            nace_code=nace_code,
            nace_description=nace_description,
            user_query=query
        )

        response = ollama.chat(model="llama3", messages=[{"role": "user", "content": prompt}])
        model_output = response['message']['content']

        # Save in required format
        output_records.append({
            "prompt": query,
            "expected_completion": expected_completion,
            "model_output": model_output
        })

    except Exception as e:
        print(f"Error on query: {query}\n{e}")
        continue

  0%|          | 0/159 [00:00<?, ?it/s]

100%|██████████| 159/159 [42:24<00:00, 16.00s/it]


In [40]:
# === Save outputs for later evaluation ===
with open(f"C:\\Users\\Giorgos\\Desktop\\HMMY\\10ο Εξάμηνο\\Διπλωματική\\8. LLM - RAG\\Extracted Files\\Inference Outputs\\inference_outputs_20.jsonl", "w", encoding="utf-8") as f:
    for entry in output_records:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")